# A2c -- functional co-clustering of co-detected genes

Analysis **A2c** of `plans/Round2_response_analysis_plan.md`.

### What this answers

Reviewer #2, major point 6:

> *"because detection seeds on individual marker genes, each granule essentially expresses just
> the single marker it was detected on … The granule subtypes … therefore largely reflect the
> seeding marker rather than any genuine multi-gene granule transcriptome"*

The rebuttal is to show that the part of a granule's content mcDETECT **never touches** -- the 270
non-seed genes -- is structured and non-random.

Note this was not something the reviewer asked for; the co-clustering idea is our own action note
under that point. The reviewer's claim above is the thing to disprove.

### Independent of A2a and A2b

- **Not A2a's subset.** Conditioning on ≥3 unique genes would select on the very statistic being
  measured. This runs on the **full published detection**, all 1,080,146 granules. Granules with
  one gene contribute no pairs under the observed data or the null, so they cost nothing.
- **Not A2b's null.** That is a detection-level null answering a different question. Here the null
  is a degree-preserving shuffle of the granule × gene table, computable now.

### The circularity trap

`merge_sphere()` merges overlapping spheres seeded by *different* markers, so co-occurrence among
the 20 seed markers is partly manufactured by detection -- 64.7 % of granules carry ≥2 of them.
The primary analysis is therefore **non-seed genes only**. The seed arm is still computed, as a
**positive control** for the statistic, and labelled detection-confounded wherever it appears.

### The contrast that makes this discriminating

Functional coherence on its own does **not** separate a granule from "any co-expressed transcript
cluster" -- coherence is what co-expression looks like. What separates them is *which kind* of
programme organises the co-occurrence. The panel's own annotation gives both kinds:

| programme | groups | granule (packaged transport) | co-expressed cluster |
|---|---|---|---|
| **localization** | pre-syn, post-syn, Neuropil, Dendrites, Axons | enriched | no reason to stand out |
| **co-expression** | 8 cell-type sets, 15 region sets, 6 layer sets | at/near chance | **enriched by definition** |

If glial and regional marker sets co-occur as strongly as synaptic ones, that is a result against
us and is reported as such.

**Runtime:** the null is an exact curveball chain over ~6 M detections; the primary arm takes
roughly 20-30 min and the full set of arms 1-2 h. Set `DRY_RUN = True` first.

**Run this notebook from `R2_revision/sparsity_structure/`.**

## 0. Setup

In [2]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))          # run from R2_revision/sparsity_structure/
import a2_config as C
import a2_common as A2

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

# -------------------- runtime gates -------------------- #
DRY_RUN = False            # True -> subsample granules and use few null draws; run this first
N_NULL = 20               # null states per arm; MC error on E is 1/sqrt(N_NULL) of the null sd
RUN_STRATIFIED = True     # section 5 robustness arms (Isocortex, per-sample, seed control)
VALIDATE = False          # section 6 correctness gates

DRY_GRANULES = 100_000
if DRY_RUN:
    N_NULL = 5

C.ensure_dirs()
OUT = C.A2C_DIR
print("writing to", OUT, "| DRY_RUN =", DRY_RUN)

writing to /Users/chenyang/Desktop/mcDETECT/R2_revision/sparsity_structure/output/a2c | DRY_RUN = False


## 1. The granule x gene matrix, and the panel's functional groups

Binarised from the published combined object's raw `counts` layer -- presence/absence, because
co-occurrence is a question about which genes are *there*, not how many copies.

In [3]:
granule_adata = sc.read_h5ad(C.COMBINED_GRANULE_ADATA)
genes = list(granule_adata.var_names)
seed_genes = [g for g in C.SYN_GENES if g in genes]
nonseed_genes = [g for g in genes if g not in set(seed_genes)]
print(f"{granule_adata.n_obs:,} granules x {len(genes)} panel genes "
      f"({len(seed_genes)} seed markers, {len(nonseed_genes)} non-seed)")

B_all = (granule_adata.layers["counts"] > 0).tocsr()
obs = granule_adata.obs

if DRY_RUN:
    rng = np.random.default_rng(0)
    sel = np.sort(rng.choice(granule_adata.n_obs, size=min(DRY_GRANULES, granule_adata.n_obs),
                             replace=False))
    B_all = B_all[sel]
    obs = obs.iloc[sel]
    print(f"DRY RUN: subsampled to {B_all.shape[0]:,} granules")

# Per-gene detection counts drive the abundance-matched permutation in section 3. Rare genes
# carry systematically higher z, and the localization groups are ~5x more abundant than the
# co-expression groups, so an unmatched test would be biased before any biology enters.
gene_detections = pd.Series(np.asarray(B_all.sum(0)).ravel(), index=genes)
print(f"detections per gene: median {gene_detections.median():,.0f}, "
      f"range {gene_detections.min():,.0f}-{gene_detections.max():,.0f}")

col = {g: i for i, g in enumerate(genes)}
idx_nonseed = np.array([col[g] for g in nonseed_genes])
idx_seed = np.array([col[g] for g in seed_genes])

1,080,146 granules x 290 panel genes (20 seed markers, 270 non-seed)
detections per gene: median 5,230, range 205-661,987


In [4]:
# The panel's own curated annotation -- both localization and co-expression programmes.
groups, dropped = A2.load_panel_annotation(
    C.panel_annotation_path("WT"), genes=genes, group_columns=C.PANEL_GROUP_COLUMNS,
    exclude_levels=C.PANEL_EXCLUDE_LEVELS, seed_genes=seed_genes, min_size=C.MIN_GROUP_SIZE)

gdf = pd.DataFrame([{k: v for k, v in g.items() if k != "genes"} for g in groups])
print(f"{len(groups)} testable groups (>= {C.MIN_GROUP_SIZE} non-seed genes):\n")
for prog in ["localization", "co-expression"]:
    sub = gdf[gdf["programme"] == prog].sort_values("n_genes", ascending=False)
    print(f"  {prog}:")
    for _, r in sub.iterrows():
        print(f"    {r['group']:24s} ({r['column']:18s}) {r['n_genes']:3d} genes, "
              f"{r['n_pairs']:5d} pairs")

dropped.to_csv(OUT / "groups_dropped.csv", index=False)
gdf.to_csv(OUT / "groups_tested.csv", index=False)
if len(dropped):
    print(f"\ndropped for having < {C.MIN_GROUP_SIZE} non-seed genes (recorded, not hidden):")
    print(dropped.to_string(index=False))

28 testable groups (>= 4 non-seed genes):

  localization:
    Neuropil                 (Neuropil          )  39 genes,   741 pairs
    pre-syn                  (Synapse markers   )  19 genes,   171 pairs
    post-syn                 (Synapse markers   )  16 genes,   120 pairs
    Axons                    (Neuropil          )   4 genes,     6 pairs
  co-expression:
    TH                       (Region markers    )  10 genes,    45 pairs
    Inhibitory neurons       (Cell type markers )  10 genes,    45 pairs
    Microglia                (Cell type markers )  10 genes,    45 pairs
    STR                      (Region markers    )  10 genes,    45 pairs
    FT                       (Region markers    )  10 genes,    45 pairs
    Astrocytes               (Cell type markers )   9 genes,    36 pairs
    CTXSp                    (Region markers    )   9 genes,    36 pairs
    Excitatory neurons       (Cell type markers )   9 genes,    36 pairs
    Isocortex_Layer1         (Region markers    

## 2. Co-occurrence enrichment

The null holds **both** margins exactly -- every granule keeps its number of distinct genes, and
every gene keeps the number of granules it appears in. Those are precisely the two effects that
would otherwise look like co-occurrence: complex granules pair everything with everything, and
abundant genes pair with everything.

The obvious analytic shortcut (a maximum-entropy model fixing the degrees only *in expectation*)
is badly biased for data this sparse -- a granule with exactly *k* genes contributes exactly
`C(k,2)` pairs, while a soft-degree null contributes about `k^2/2`, a factor of `k/(k-1)`, which
at the median complexity of ~5 genes is a ~25 % over-estimate on **every** pair. Curveball has no
such bias because its constraints are hard. Section 6 checks the calibration rather than assuming
it.

In [5]:
def run_arm(name, row_mask, gene_idx, gene_list, n_null, note=""):
    t0 = time.time()
    Bm = B_all[row_mask][:, gene_idx] if row_mask is not None else B_all[:, gene_idx]
    print(f"\n=== arm '{name}' : {Bm.shape[0]:,} granules x {Bm.shape[1]} genes {note}", flush=True)
    pairs, Z, info = A2.cooccurrence_enrichment(Bm, gene_list, n_null=n_null, seed=0)
    pairs["arm"] = name
    info["arm"] = name
    info["minutes"] = round((time.time() - t0) / 60, 2)
    print(f"    done in {info['minutes']} min | median z = {pairs['z'].median():.3f}", flush=True)
    return pairs, Z, info


pairs_primary, Z_primary, info_primary = run_arm(
    "all", None, idx_nonseed, nonseed_genes, N_NULL, note="(non-seed, primary)")


=== arm 'all' : 1,080,146 granules x 270 genes (non-seed, primary)
  null: 593,195 granules, 3,257,809 detections, burn-in 16,289,045 trades, then 20 states spaced 3,257,809 apart
    null sample 1/20
    null sample 2/20
    null sample 3/20
    null sample 4/20
    null sample 5/20
    null sample 6/20
    null sample 7/20
    null sample 8/20
    null sample 9/20
    null sample 10/20
    null sample 11/20
    null sample 12/20
    null sample 13/20
    null sample 14/20
    null sample 15/20
    null sample 16/20
    null sample 17/20
    null sample 18/20
    null sample 19/20
    null sample 20/20
    done in 4.87 min | median z = 0.347


## 3. Group-level test -- localization versus co-expression

Statistic: the median `z` over within-group pairs. Significance by permuting the **gene → group**
assignment across all non-seed genes, preserving group size -- pairs share genes and are not
independent, so a test treating them as independent observations would be badly
anti-conservative.

In [6]:
group_res = A2.group_enrichment_test(Z_primary, nonseed_genes, groups,
                                     gene_weights=gene_detections.reindex(nonseed_genes).to_numpy(),
                                     n_perm=(200 if DRY_RUN else C.GROUP_TEST_N_PERM),
                                     seed=C.GROUP_TEST_SEED)
group_res["arm"] = "all"
print(group_res[["programme", "group", "n_genes", "n_pairs", "median_z", "null_median_z",
                 "ses", "p_upper", "q_upper_star"]].to_string(index=False))

summary = (group_res.groupby("programme")["median_z"]
           .agg(["count", "median", "min", "max"]).reset_index())
print("\nby programme:")
print(summary.to_string(index=False))
print(f"\nbackground median z over all non-seed pairs: {group_res['background_median_z'].iloc[0]:.3f}")

    programme                 group  n_genes  n_pairs  median_z  null_median_z       ses  p_upper q_upper_star
 localization                 Axons        4        6 37.259790      -1.876812  4.663062 0.002499           **
co-expression            Astrocytes        9       36 10.529778      -0.569481  7.565471 0.000500           **
co-expression             Microglia       10       45 10.493110       0.455330 13.822669 0.000500           **
co-expression                   OPC        6       15  9.017985       1.111603  5.784792 0.001000           **
co-expression      Oligodendrocytes        7       21  7.922548       0.972443  4.792799 0.001000           **
co-expression    Inhibitory neurons       10       45  7.785950       0.085546  4.245039 0.001000           **
co-expression         Isocortex_RSP        4        6  7.505034       1.543598  3.178441 0.006497            *
 localization               pre-syn       19      171  5.379229      -0.788532  4.045040 0.001499           **
c

## 4. Block structure

Hierarchical clustering of the 270 x 270 enrichment matrix, with a programme colour bar. The
question is whether a coherent localization block appears -- not whether every pair is
significant.

In [7]:
gene_to_prog = {}
for g in groups:
    for gene in g["genes"]:
        gene_to_prog.setdefault(gene, g["programme"])
prog_colors = pd.Series([gene_to_prog.get(g, "unannotated") for g in nonseed_genes],
                        index=nonseed_genes, name="Programme").map(
    {**C.PROGRAMME_COLORS, "unannotated": "#e8e8e8"})

Zc = pd.DataFrame(Z_primary, index=nonseed_genes, columns=nonseed_genes)
Zc = Zc.fillna(0.0)
vmax = float(np.nanpercentile(np.abs(Z_primary), 99))
g = sns.clustermap(Zc, cmap="bwr", center=0, vmin=-vmax, vmax=vmax,
                   row_colors=prog_colors, col_colors=prog_colors,
                   xticklabels=False, yticklabels=False, figsize=(9, 9),
                   cbar_pos=(0.02, 0.83, 0.03, 0.12))
g.ax_heatmap.set_xlabel("Non-seed panel genes")
g.ax_heatmap.set_ylabel("Non-seed panel genes")
g.cax.set_title("z", fontsize=10)
plt.savefig(OUT / "cooccurrence_clustermap.jpeg", dpi=500, bbox_inches="tight")
plt.close()
print("saved cooccurrence_clustermap.jpeg")

order = [nonseed_genes[i] for i in g.dendrogram_row.reordered_ind]
pd.DataFrame({"gene": order,
              "programme": [gene_to_prog.get(x, "unannotated") for x in order]}
             ).to_csv(OUT / "clustermap_gene_order.csv", index=False)

saved cooccurrence_clustermap.jpeg


## 5. Robustness arms

Each is a separate column in the same tables, not a re-run of the argument.

* **Isocortex only** -- regional marker groups should collapse here, which is the built-in check
  that the regional confound is what we think it is.
* **WT and AD separately** -- does the structure hold in both samples?
* **Seed markers** -- the same statistic on the 20 detection seeds. This is a *positive control*:
  `merge_sphere` merges spheres from different markers, so these must come out enriched. It is
  not evidence for anything, and is labelled detection-confounded everywhere it appears.

In [8]:
all_pairs = [pairs_primary]
all_groups = [group_res]
all_info = [info_primary]

if RUN_STRATIFIED:
    arms = [
        ("Isocortex", (obs["brain_area"].to_numpy() == C.ROI), idx_nonseed, nonseed_genes,
         "(non-seed, within Isocortex)"),
        ("WT", (obs["batch"].to_numpy() == C.dataset("WT")), idx_nonseed, nonseed_genes,
         "(non-seed, WT only)"),
        ("AD", (obs["batch"].to_numpy() == C.dataset("AD")), idx_nonseed, nonseed_genes,
         "(non-seed, AD only)"),
        ("seed_markers", None, idx_seed, seed_genes,
         "(SEED markers -- positive control, detection-confounded)"),
    ]
    for name, mask, gi, gl, note in arms:
        p_, Z_, i_ = run_arm(name, mask, gi, gl, N_NULL, note=note)
        all_pairs.append(p_)
        all_info.append(i_)
        if name != "seed_markers":
            gr = A2.group_enrichment_test(Z_, gl, groups,
                                          gene_weights=gene_detections.reindex(gl).to_numpy(),
                                          n_perm=(200 if DRY_RUN else C.GROUP_TEST_N_PERM),
                                          seed=C.GROUP_TEST_SEED)
            gr["arm"] = name
            all_groups.append(gr)


=== arm 'Isocortex' : 334,361 granules x 270 genes (non-seed, within Isocortex)
  null: 202,383 granules, 1,114,926 detections, burn-in 5,574,630 trades, then 20 states spaced 1,114,926 apart
    null sample 1/20
    null sample 2/20
    null sample 3/20
    null sample 4/20
    null sample 5/20
    null sample 6/20
    null sample 7/20
    null sample 8/20
    null sample 9/20
    null sample 10/20
    null sample 11/20
    null sample 12/20
    null sample 13/20
    null sample 14/20
    null sample 15/20
    null sample 16/20
    null sample 17/20
    null sample 18/20
    null sample 19/20
    null sample 20/20
    done in 1.67 min | median z = 0.052

=== arm 'WT' : 681,337 granules x 270 genes (non-seed, WT only)
  null: 367,816 granules, 2,052,371 detections, burn-in 10,261,855 trades, then 20 states spaced 2,052,371 apart
    null sample 1/20
    null sample 2/20
    null sample 3/20
    null sample 4/20
    null sample 5/20
    null sample 6/20
    null sample 7/20
    null sa

In [9]:
pairs_out = pd.concat(all_pairs, ignore_index=True)
pairs_out["pair_is_seed"] = pairs_out["arm"].eq("seed_markers")
pairs_out.to_parquet(OUT / "pair_enrichment.parquet", index=False)

groups_out = pd.concat(all_groups, ignore_index=True)
groups_out.to_csv(OUT / "group_enrichment.csv", index=False)

info_out = pd.DataFrame(all_info)
info_out["dry_run"] = DRY_RUN
info_out["n_null"] = N_NULL
info_out.to_csv(OUT / "run_info.csv", index=False)

print(f"wrote {len(pairs_out):,} pair rows across {pairs_out['arm'].nunique()} arms")
print(info_out.to_string(index=False))
if DRY_RUN:
    print("\nDRY RUN -- numbers are not final. Set DRY_RUN = False and re-run.")

wrote 145,450 pair rows across 5 arms
 n_granules_used  n_granules_dropped  n_genes  n_null_samples     nnz  burnin_trades  spacing_trades  mc_error_frac_of_sd          arm  minutes  dry_run  n_null
          593195              486951      270              20 3257809       16289045         3257809             0.223607          all     4.87    False      20
          202383              131978      270              20 1114926        5574630         1114926             0.223607    Isocortex     1.67    False      20
          367816              313521      270              20 2052371       10261855         2052371             0.223607           WT     3.37    False      20
          225379              173430      270              20 1205438        6027190         1205438             0.223607           AD     1.91    False      20
          698526              381620       20              20 2317328       11586640         2317328             0.223607 seed_markers     2.59    False     

## 6. Correctness gates

Off by default. These check the statistic, not the biology -- and the first one is not optional
in spirit: an uncalibrated `z` would inflate every claim above.

In [10]:
if VALIDATE:
    from scipy.sparse import csr_matrix

    # (a) z is calibrated. Feed the estimator a matrix drawn FROM its own null -- shuffle the real
    #     data hard, then score it. z must come out ~N(0,1); anything else means E or Var is wrong.
    sub = B_all[:50_000][:, idx_nonseed].tocsr()
    rows = [set(sub.indices[sub.indptr[i]:sub.indptr[i + 1]].tolist()) for i in range(sub.shape[0])]
    A2.curveball(rows, 10 * sub.nnz, len(nonseed_genes), np.random.default_rng(99))
    null_draw = A2._rows_to_csr(rows, len(nonseed_genes))
    pv, Zv, _ = A2.cooccurrence_enrichment(null_draw, nonseed_genes, n_null=20, seed=1,
                                           verbose=False)
    zv = pv["z"].to_numpy(); zv = zv[np.isfinite(zv)]
    cal = {"mean": float(zv.mean()), "sd": float(zv.std()),
           "frac_abs_gt_1.96": float(np.mean(np.abs(zv) > 1.96))}
    print("(a) z on a true null draw:", {k: round(v, 4) for k, v in cal.items()},
          "-- expect mean~0, sd~1, frac~0.05")
    assert abs(cal["mean"]) < 0.25 and 0.8 < cal["sd"] < 1.3, "z is not calibrated"

    # (b) the null really does preserve both margins.
    r0 = np.asarray(sub.sum(1)).ravel(); c0 = np.asarray(sub.sum(0)).ravel()
    r1 = np.asarray(null_draw.sum(1)).ravel(); c1 = np.asarray(null_draw.sum(0)).ravel()
    print("(b) margins preserved -- rows:", np.array_equal(np.sort(r0), np.sort(r1)),
          "cols:", np.array_equal(c0, c1))
    assert np.array_equal(c0, c1)

    # (c) positive control: the seed markers are merged across spheres by construction, so they
    #     must show clear enrichment. If they do not, the statistic is not detecting real
    #     structure and nothing above is trustworthy.
    seed_pairs = pairs_out[pairs_out["arm"] == "seed_markers"]
    if len(seed_pairs):
        print(f"(c) seed-marker median z = {seed_pairs['z'].median():.2f} "
              f"(non-seed primary = {pairs_primary['z'].median():.2f})")

    # (d) the group test is calibrated: random gene sets must give ~uniform p-values.
    rng = np.random.default_rng(1)
    rand_groups = [{"column": "random", "group": f"r{i}", "programme": "co-expression",
                    "genes": [nonseed_genes[j] for j in
                              rng.choice(len(nonseed_genes), 10, replace=False)]}
                   for i in range(100)]
    rg = A2.group_enrichment_test(Z_primary, nonseed_genes, rand_groups,
                                  gene_weights=gene_detections.reindex(nonseed_genes).to_numpy(),
                                  n_perm=500, seed=2)
    print(f"(d) random groups: mean p {rg['p_upper'].mean():.3f} (~0.5), "
          f"frac p<0.05 {np.mean(rg['p_upper'] < 0.05):.3f} (~0.05)")

    pd.DataFrame([cal]).to_csv(OUT / "null_calibration.csv", index=False)